# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [ ]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [5]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif

num_col = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'] #define the columns
for col in num_col:
    spaceship[col] = spaceship[col].fillna(spaceship[col].median()) #clean the nuls

# split in Y and X
X = spaceship[num_col]
y = spaceship['Transported'].astype(int)

#feature scaling
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Feature scaling")
print(X_scaled.head())

#feature selection
k_best_selector = SelectKBest(score_func=f_classif, k=4)
X_selected = k_best_selector.fit_transform(X_scaled, y)
selected_features = X_scaled.columns[k_best_selector.get_support()]

print("Feature selection")
print(list(selected_features))

df_final = pd.DataFrame(X_selected, columns=selected_features)

Feature scaling
        Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck
0  0.493671     0.000000   0.000000      0.000000  0.000000  0.000000
1  0.303797     0.007608   0.000302      0.001064  0.024500  0.001823
2  0.734177     0.003001   0.119948      0.000000  0.299670  0.002030
3  0.417722     0.000000   0.043035      0.015793  0.148563  0.007997
4  0.202532     0.021149   0.002348      0.006428  0.025214  0.000083
Feature selection
['Age', 'RoomService', 'Spa', 'VRDeck']


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df_final, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

gb_model = GradientBoostingClassifier(
    n_estimators=100,      
    learning_rate=0.1,    
    max_depth=3,         
    random_state=42
)
gb_model.fit(X_train, y_train) #train the model
y_pred_gb = gb_model.predict(X_test) #prediction
acc_gb = accuracy_score(y_test, y_pred_gb) #accuracy

print("Accuracy")
print(f"Accuracy : {acc_gb:.4f}\n")

print("Report")
print(classification_report(y_test, y_pred_gb))

Accuracy
Accuracy : 0.7752

Report
              precision    recall  f1-score   support

           0       0.80      0.73      0.76       863
           1       0.76      0.82      0.79       876

    accuracy                           0.78      1739
   macro avg       0.78      0.77      0.77      1739
weighted avg       0.78      0.78      0.77      1739



- Evaluate your model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not transported, Transported'],
            yticklabels=['Not transported, Transported'])

plt.title('Confusion Matrix')
plt.xlabel('Prediction')
plt.ylabel('Real Value')

print("Accuracy")
print(f"Accuracy : {acc_gb:.4f}\n")

print("Report")
print(classification_report(y_test, y_pred_gb))

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0]
}

- Run Grid Search

In [ ]:
grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,                      # Validación cruzada de 5 pliegues
    scoring='accuracy',        # Métrica para optimizar
    n_jobs=-1                  # Usar todos los procesadores para ir más rápido
)

grid_search.fit(X_train, y_train)

best_gb_model = grid_search.best_estimator_

print(grid_search.best_params_)

- Evaluate your model

In [ ]:
y_pred_tuned = best_gb_model.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"\nAccuracy : {acc_tuned:.4f}")

-.